In [ ]:
# Setup the Jupyter version of Dash
from dash import Dash

# Dashboard imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64



# Additional imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# Connect to MongoDB securely using environment variables
db = AnimalShelter()

# Load animal records from MongoDB
# Include the full dataset so rescue filters can find matches
df = pd.DataFrame.from_records(
    db.read({})
)

# Remove MongoDB ObjectId because Dash cannot display it
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)
# Convert imported CSV column names to dashboard-compatible names
df.columns = [
    column.strip().lower().replace(" ", "_")
    for column in df.columns
]
print("Total records loaded:", len(df))
print("Available columns:", df.columns.tolist())
print(df.head(3))
#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'

# If your image has a different filename, change the line above.
encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
).decode()


app.layout = html.Div([

    # Logo
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={
                'height': '200px',
                'marginBottom': '10px'
            }
        )
    ),

    # Dashboard title
    html.Center(
        html.H1('Grazioso Salvare Rescue Dashboard')
    ),

    # Unique identifier required by rubric
    html.Center(
        html.H3('Created by Matthew Lambert')
    ),

    html.Hr(),

    # Interactive filter
    html.Div([
        html.H3('Rescue Type Filter'),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {
                    'label': ' Water Rescue',
                    'value': 'Water'
                },
                {
                    'label': ' Mountain or Wilderness Rescue',
                    'value': 'Mountain'
                },
                {
                    'label': ' Disaster or Individual Tracking',
                    'value': 'Disaster'
                },
                {
                    'label': ' Reset',
                    'value': 'Reset'
                }
            ],
            value='Reset',
            labelStyle={
                'display': 'inline-block',
                'marginRight': '25px'
            }
        )
    ]),

    html.Hr(),

    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        # Table features
        page_size=10,

        sort_action='native',
        sort_mode='multi',

        filter_action='native',

        row_selectable='single',

        selected_rows=[0],

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '250px',
            'whiteSpace': 'normal'
        },

        style_header={
            'fontWeight': 'bold'
        }
    ),

    html.Br(),
    html.Hr(),

    # Pie chart and map displayed side by side
    html.Div(
        className='row',

        style={
            'display': 'flex',
            'flexWrap': 'wrap'
        },

        children=[

            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={
                    'width': '50%'
                }
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={
                    'width': '50%'
                }
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################


# Update table based on rescue filter
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    # Water Rescue
    if filter_type == 'Water':

        query = {
            "Animal Type": "Dog",

            "Breed": {
    "$in": [
        "Labrador Retriever",
        "Labrador Retriever Mix",
        "Chesapeake Bay Retriever",
        "Newfoundland"
    ]
},

            "Sex upon Outcome": "Intact Female",

            "Age upon Outcome": {
    "$in": [
        "6 months",
        "7 months",
        "8 months",
        "9 months",
        "10 months",
        "11 months",
        "1 year",
        "2 years",
        "3 years"
    ]
}
        }

    # Mountain or Wilderness Rescue
    elif filter_type == 'Mountain':

        query = {
            "Animal Type": "Dog",

            "Breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "Sex upon Outcome": "Intact Male",

            "Age upon Outcome": {
                 "$in": [
                     "6 months",
                    "7 months",
                     "8 months",
                     "9 months",
                    "10 months",
                    "11 months",
                    "1 year",
                    "2 years",
                    "3 years"
             ]
}
        }

    # Disaster or Individual Tracking
    elif filter_type == 'Disaster':

        query = {
            "Animal Type": "Dog",

            "Breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "Sex upon Outcome": "Intact Male",

            "Age upon Outcome": {
                "$in": [
                    "5 months",
                    "6 months",
                    "7 months",
                    "8 months",
                    "9 months",
                    "10 months",
                    "11 months",
                    "1 year",
                    "2 years",
                    "3 years",
                    "4 years",
                    "5 years"
                ]
}
        }

    # Reset
    else:
        query = {}

   
    # Remove MongoDB _id
    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff = dff.drop(columns=['_id'])

    # Convert MongoDB field names to dashboard column names
    dff.columns = [
        column.strip().lower().replace(" ", "_")
        for column in dff.columns
    ]

    return dff.to_dict('records')



#############################################
# Pie Chart
#############################################

@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    # If no filtered data exists, use original dataframe
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Avoid errors if there are no records
    if dff.empty:
        return [
            html.H3("No animals match this filter.")
        ]

    # Count animals by breed
    breed_counts = (
        dff['breed']
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    figure = px.pie(
        breed_counts,
        values='count',
        names='breed',
        title='Animals by Breed'
    )

    return [
        dcc.Graph(
            figure=figure
        )
    ]


#############################################
# Highlight Selected Columns
#############################################

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):

    if not selected_columns:
        return []

    return [
        {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        }
        for i in selected_columns
    ]


#############################################
# Geolocation Map
#############################################

@app.callback(
    Output('map-id', 'children'),

    [
        Input(
            'datatable-id',
            'derived_virtual_data'
        ),

        Input(
            'datatable-id',
            'derived_virtual_selected_rows'
        )
    ]
)
def update_map(viewData, index):

    # Use original dataframe if table has not yet changed
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Prevent map errors if no animals match
    if dff.empty:
        return [
            html.H3(
                "No location data available."
            )
        ]

    # Default to first row
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Protect against row index errors
    if row >= len(dff):
        row = 0

    # Get selected animal
    animal = dff.iloc[row]

    # Austin, Texas reference coordinates
# Individual animal locations are not available in the dataset
    latitude = 30.2672
    longitude = -97.7431

    breed = animal['breed']
    name = animal['name']

    # Austin, Texas area map
    return [
        dl.Map(

            style={
                'width': '100%',
                'height': '500px'
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[

                dl.TileLayer(),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            breed
                        ),

                        dl.Popup([
                            html.H3(
                                "Animal Information"
                            ),

                            html.P(
                                "Name: {}".format(name)
                            ),

                            html.P(
                                "Breed: {}".format(breed)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


#########################
# Run Dashboard
#########################

app.run(jupyter_mode="external", debug=False)